# Where does QUANT's advantage over other interval methods come from?

QUANT (Dempster, Schmidt & Webb, DMKD 2024) reports a mean accuracy of **0.8671** over 112 UCR datasets, ahead of DrCIF (0.8636) and rSTSF (0.8640), with win/draw/loss of 65/4/43 and 65/5/42.

Those are single numbers averaged over a very heterogeneous archive. This notebook asks two questions:

1. Is the advantage **uniform across domains**, or concentrated?
2. How **robust** is it to dataset selection?

The paper itself invites the second question (Sect. 4.1):

> the results for all three methods are highly correlated, and the differences in accuracy are mostly small, even small changes in accuracy could change the appearance of the results, in particular, the ratio of wins and losses

### Sources

| what | where | note |
|---|---|---|
| QUANT per-resample accuracies | `Quant/.../results/accuracy_resamples_ucr142.csv` | shipped with the paper's repo |
| baseline accuracies | `aeon.benchmarking.published_results` | bake-off redux |
| our own reproduction | `raw_results.csv` | 112 datasets x 30 resamples, aeon resampling |
| domain labels | `aeon.datasets.get_dataset_meta_data` | from timeseriesclassification.com |

Sections 2 onwards use **published** numbers only, so the findings are about the paper rather than about our reproduction. Our own run appears in section 3 purely as a cross-check. No new compute is needed.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

from aeon.datasets import get_dataset_meta_data, tsc_datasets
from aeon.benchmarking.published_results import (
    load_classification_bake_off_2023_results,
)

pd.set_option("display.width", 140)
pd.set_option("display.max_rows", 200)

import os
from pathlib import Path

ROOT = Path(os.environ.get("QUANT_ROOT", Path.cwd().parent))
PUBLISHED = f"{ROOT}/Quant/quant-main/quant-main/results/accuracy_resamples_ucr142.csv"
MINE = f"{ROOT}/results/raw_results.csv"

BASELINES = ["DrCIF", "RSTSF", "CIF", "TSF"]   # the interval methods in Fig. 1
STRONG = ["DrCIF", "RSTSF"]                     # the two QUANT is closest to

# STSF is the fifth method in Fig. 1 but is not present in aeon's copy of the
# bake-off results, so it is omitted throughout.

## 1. Assemble the data

One row per dataset: QUANT's published mean, our own mean, each baseline's mean, and the domain label.

`aeon` returns no metadata row for two of the 112 datasets, which are handled differently:

- **MixedShapesRegularTrain** — labelled `IMAGE`, taken directly from its entry on timeseriesclassification.com.
- **InsectWingbeatSound** — assigned `AUDIO` as a **judgement call**, not from a source. The data originates as insect wingbeat sound but what the archive stores is the power spectrum of that sound, so the category is genuinely ambiguous between audio, sensor and spectrogram. It lands in a two-dataset group and carries no weight in any conclusion below; section 7 repeats the key analysis without it.

In [2]:
FROM_WEBSITE = {"MixedShapesRegularTrain": "IMAGE"}
JUDGEMENT_CALL = {"InsectWingbeatSound": "AUDIO"}
MISSING = {**FROM_WEBSITE, **JUDGEMENT_CALL}

names = sorted(tsc_datasets.univariate_equal_length)
meta = get_dataset_meta_data(names)[
    ["Dataset", "Type", "TrainSize", "TestSize", "Length", "NumberClasses"]
]

for name, type_ in MISSING.items():
    if name not in set(meta["Dataset"]):
        meta.loc[len(meta)] = [name, type_, None, None, None, None]

meta = meta.set_index("Dataset")

# the None fill above makes these columns object dtype; cast them back
for col in ["TrainSize", "TestSize", "Length", "NumberClasses"]:
    meta[col] = pd.to_numeric(meta[col], errors="coerce")

print(f"{len(meta)} datasets labelled")
meta["Type"].value_counts()

112 datasets labelled


Type
IMAGE           32
SENSOR          16
SPECTRO         12
HAR             11
SIMULATED        9
DEVICE           9
ECG              7
MOTION           6
HEMODYNAMICS     3
AUDIO            2
EPG              2
EOG              2
TRAFFIC          1
Name: count, dtype: int64

In [3]:
def load_published(path):
    df = pd.read_csv(path)
    out = df.set_index(df.columns[0])
    out.columns = range(len(out.columns))
    return out


published = load_published(PUBLISHED)
results = load_classification_bake_off_2023_results()

ours = (
    pd.read_csv(MINE)
    .drop_duplicates(["dataset", "resample"])
    .groupby("dataset")["accuracy"]
    .mean()
)

frame = {"QUANT": published.mean(axis=1), "QUANT_ours": ours}
for m in BASELINES:
    frame[m] = pd.Series({d: np.mean(v) for d, v in results[m].items()})

df = pd.DataFrame(frame).join(meta, how="inner").dropna(subset=["QUANT"] + BASELINES)
print(f"{len(df)} datasets with results for every method")
df.to_csv(f"{ROOT}/results/results_with_baselines.csv")
df.head()

112 datasets with results for every method


,QUANT,QUANT_ours,DrCIF,RSTSF,CIF,TSF,Type,TrainSize,TestSize,Length,NumberClasses
ACSF1,0.807667,0.810000,0.782667,0.820333,0.766667,0.667000,DEVICE,100.0,100.0,1460.0,10.0
Adiac,0.812788,0.810315,0.811679,0.811935,0.766922,0.715686,IMAGE,390.0,391.0,176.0,37.0
ArrowHead,0.829143,0.844381,0.828000,0.826857,0.823238,0.784190,IMAGE,36.0,175.0,251.0,3.0
BME,1.000000,1.000000,0.998889,0.999778,0.997333,0.960444,SIMULATED,30.0,150.0,128.0,3.0
Beef,0.785556,0.778889,0.794444,0.792222,0.792222,0.677778,SPECTRO,30.0,30.0,470.0,5.0


## 2. Sanity check, and a point the paper understates

First confirm the assembled table reproduces the paper's own headline comparison. If it matches Fig. 1, everything below can be trusted.

In [4]:
def wdl(a, b):
    return f"{int((a > b).sum())}/{int((a == b).sum())}/{int((a < b).sum())}"


PAPER_FIG1 = {  # mean accuracy, W/D/L and Wilcoxon p as printed in Fig. 1
    "DrCIF": (0.8636, "65/4/43", "0.1442"),
    "RSTSF": (0.8640, "65/5/42", "0.0277"),
    "CIF": (0.8478, "82/2/28", "<=1e-04"),
    "TSF": (0.8022, "106/3/3", "<=1e-04"),
}

q = df["QUANT"]
rows = []
for m in BASELINES:
    paper_acc, paper_wdl, paper_p = PAPER_FIG1[m]
    rows.append(
        {
            "baseline": m,
            "acc here": round(df[m].mean(), 4),
            "acc paper": paper_acc,
            "QUANT - them": round(q.mean() - df[m].mean(), 4),
            "W/D/L here": wdl(q, df[m]),
            "W/D/L paper": paper_wdl,
            "p here": round(wilcoxon(q, df[m], zero_method="wilcox").pvalue, 4),
            "p here (pratt)": round(wilcoxon(q, df[m], zero_method="pratt").pvalue, 4),
            "p paper": paper_p,
        }
    )

print(f"QUANT mean here: {q.mean():.4f}   paper: 0.8671\n")
pd.DataFrame(rows)

QUANT mean here: 0.8671   paper: 0.8671



,baseline,acc here,acc paper,QUANT - them,W/D/L here,W/D/L paper,p here,p here (pratt),p paper
0,DrCIF,0.8636,0.8636,0.0035,65/4/43,65/4/43,0.1595,0.1442,0.1442
1,RSTSF,0.8640,0.8640,0.0031,65/5/42,65/5/42,0.0294,0.0277,0.0277
2,CIF,0.8478,0.8478,0.0193,82/2/28,82/2/28,0.0000,0.0000,<=1e-04
3,TSF,0.8022,0.8022,0.0650,106/3/3,106/3/3,0.0000,0.0000,<=1e-04


Means and win/draw/loss match Fig. 1 exactly. The p-values match exactly under `zero_method="pratt"`; scipy's default (`"wilcox"`), which drops zero differences, gives slightly larger values (0.1595 and 0.0294). The table is a faithful reconstruction.

**That reconstruction makes something explicit that the paper's prose glosses over.** The paper states:

> Quant is more accurate on average than existing interval methods, including DrCIF and rSTSF

But of the two methods QUANT is actually close to, only one comparison is significant, and only before correction:

- **vs DrCIF: p = 0.1442** (paper), 0.1595 (here) — *not* significant at 0.05. On the paper's own test, QUANT and DrCIF are indistinguishable over these 112 datasets.
- **vs rSTSF: p = 0.0277** (paper), 0.0294 (here) — significant, but only just, and uncorrected for the comparisons made in Fig. 1. A Holm correction raises the paper's 0.0277 to 0.0553 (0.0587 from the `"wilcox"` value here), neither of which clears 0.05. The result is the same whether or not STSF is included, since its own p ≤ 1e-4.

The highly significant results (CIF, TSF, and STSF in the paper) are against the *older, weaker* interval methods, which nobody disputes QUANT beats.

So the claim that survives scrutiny is narrower than the prose suggests: QUANT **matches** the strongest existing interval methods at a fraction of the compute, rather than beating them. Which is, in fairness, what the abstract claims — *the same accuracy, on average, as the most accurate existing interval methods*.

## 3. Accuracy by domain, and a check on our own reproduction

How does QUANT do on each kind of data? And does *our* reproduction error depend on the domain? It shouldn't, but it is worth checking, since we used `aeon` resampling rather than the official Java indices.

In [5]:
by_type = df.groupby("Type").apply(
    lambda g: pd.Series(
        {
            "n": len(g),
            "QUANT": g["QUANT"].mean(),
            "ours": g["QUANT_ours"].mean(),
            "repro err": (g["QUANT_ours"] - g["QUANT"]).abs().mean(),
        }
    ),
    include_groups=False,
)
by_type["n"] = by_type["n"].astype(int)
by_type.sort_values("QUANT").round(4)

,n,QUANT,ours,repro err
Type,,,,
AUDIO,2,0.5155,0.5142,0.0013
HEMODYNAMICS,3,0.6577,0.6646,0.0093
MOTION,6,0.7318,0.7271,0.0072
EOG,2,0.8126,0.8099,0.0031
IMAGE,32,0.8453,0.8450,0.0031
DEVICE,9,0.8476,0.8459,0.0025
HAR,11,0.8824,0.8817,0.0022
SPECTRO,12,0.8937,0.8945,0.0063
SENSOR,16,0.9208,0.9211,0.0032


Accuracy varies widely by domain, but that mostly reflects how hard each problem family is rather than anything about QUANT.

Reproduction error is uniformly small and shows no domain pattern, which supports treating our run as equivalent to the published one despite the different resampling scheme.

## 4. *Advantage* by domain

The question that matters: is QUANT ahead of the other interval methods **everywhere**, or only in places?

In [6]:
rows = []
for type_, g in df.groupby("Type"):
    row = {"type": type_, "n": len(g), "QUANT": g["QUANT"].mean()}
    for m in BASELINES:
        row[f"vs {m}"] = g["QUANT"].mean() - g[m].mean()
    row["vs strong"] = np.mean([row[f"vs {m}"] for m in STRONG])
    row["wdl DrCIF"] = wdl(g["QUANT"], g["DrCIF"])
    rows.append(row)

adv = pd.DataFrame(rows).set_index("type").sort_values("vs strong", ascending=False)
adv["n"] = adv["n"].astype(int)
adv.round(4)

,n,QUANT,vs DrCIF,vs RSTSF,vs CIF,vs TSF,vs strong,wdl DrCIF
type,,,,,,,,
HEMODYNAMICS,3,0.6577,0.0955,0.0243,0.1963,0.3189,0.0599,3/0/0
EOG,2,0.8126,0.0237,0.0543,0.0393,0.1343,0.0390,2/0/0
DEVICE,9,0.8476,0.0116,0.0122,0.0308,0.1023,0.0119,7/0/2
SIMULATED,9,0.9592,0.0070,0.0015,0.0184,0.0850,0.0042,7/0/2
SENSOR,16,0.9208,0.0036,0.0023,0.0189,0.0436,0.0029,8/1/7
HAR,11,0.8824,-0.0032,0.0080,0.0020,0.0386,0.0024,5/1/5
IMAGE,32,0.8453,-0.0001,0.0043,0.0143,0.0482,0.0021,18/0/14
ECG,7,0.9500,-0.0004,0.0016,0.0145,0.0421,0.0006,4/0/3
EPG,2,1.0000,0.0000,0.0000,0.0064,0.0000,0.0000,0/2/0


**The advantage is not uniform.**

On the three largest groups — IMAGE (32 datasets), SENSOR (16), SPECTRO (12), together 60 of 112 — QUANT is essentially **at parity** with DrCIF: −0.0001, +0.0036 and −0.0013 respectively. IMAGE, nearly a third of the archive, is 18/0/14.

The clearly positive numbers come from HEMODYNAMICS (+0.0599, n=3), EOG (+0.0390, n=2) and DEVICE (+0.0119, n=9). The first two are too small to support any claim on their own.

At the other end, QUANT is behind on MOTION (−0.0159, n=6) and AUDIO (−0.0165, n=2), and slightly behind on SPECTRO (−0.0040, n=12).

## 5. A caution about merging categories

Our first pass merged ECG, HEMODYNAMICS, EOG and EPG into a single PHYSIOLOGICAL group (n=14), so that the groups would be large enough to analyse. That produced an apparently clean finding — **+0.019** over the strong baselines, the largest of any group — and a tempting story about QUANT suiting physiological signals.

Splitting it back apart dissolves it.

In [7]:
PHYSIO = ["ECG", "HEMODYNAMICS", "EOG", "EPG"]
physio = df[df["Type"].isin(PHYSIO)]

merged = np.mean([physio["QUANT"].mean() - physio[m].mean() for m in STRONG])
print(f"merged PHYSIOLOGICAL (n={len(physio)}): {merged:+.4f}\n")

display(
    adv.loc[[t for t in PHYSIO if t in adv.index], ["n", "vs strong", "wdl DrCIF"]].round(4)
)

print("\nevery dataset in the merged group:")
physio.assign(diff=physio["QUANT"] - physio["DrCIF"])[
    ["Type", "NumberClasses", "QUANT", "DrCIF", "diff"]
].sort_values("diff", ascending=False).round(4)

merged PHYSIOLOGICAL (n=14): +0.0187



,n,vs strong,wdl DrCIF
type,,,
ECG,7,0.0006,4/0/3
HEMODYNAMICS,3,0.0599,3/0/0
EOG,2,0.0390,2/0/0
EPG,2,0.0000,0/2/0



every dataset in the merged group:


,Type,NumberClasses,QUANT,DrCIF,diff
PigAirwayPressure,HEMODYNAMICS,52.0,0.4545,0.2574,0.1971
PigArtPressure,HEMODYNAMICS,52.0,0.8827,0.8341,0.0486
PigCVP,HEMODYNAMICS,52.0,0.6359,0.5952,0.0407
EOGHorizontalSignal,EOG,12.0,0.8375,0.7975,0.0400
EOGVerticalSignal,EOG,12.0,0.7878,0.7803,0.0075
ECG200,ECG,2.0,0.8690,0.8620,0.0070
NonInvasiveFetalECGThorax2,ECG,42.0,0.9537,0.9475,0.0061
TwoLeadECG,ECG,2.0,0.9872,0.9819,0.0053
NonInvasiveFetalECGThorax1,ECG,42.0,0.9355,0.9322,0.0033
InsectEPGRegularTrain,EPG,3.0,1.0000,1.0000,0.0000


ECG on its own — the category of most practical interest — shows **no advantage at all**: +0.0006 over the strong baselines, 4/0/3 against DrCIF across seven datasets.

The merged figure was carried by the three Pig datasets, and within those, overwhelmingly by `PigAirwayPressure`: QUANT 0.4545 against DrCIF 0.2574, a gap of **+0.1971**. That dataset has 52 classes and *both* methods perform badly on it; QUANT is merely less bad. It is a pathological case, not evidence of a domain strength.

**Lesson for the write-up:** a grouping choice made for statistical convenience manufactured a finding the raw labels do not support. This is worth reporting as an analysis artefact in its own right — it is exactly the kind of researcher degree of freedom that reproducibility work should surface.

## 6. How fragile is the headline result?

If single datasets can swing a group mean that much, how much do they move the overall comparison? Drop datasets in order of **absolute** QUANT−baseline gap — so wins and losses are removed alike, not just the favourable ones — and watch the mean difference, win/draw/loss and p-value.

In [8]:
def leave_out(baseline, k_max=10):
    gap = (df["QUANT"] - df[baseline]).abs().sort_values(ascending=False)
    rows = []
    for k in range(k_max + 1):
        keep = df.drop(index=gap.index[:k])
        rows.append(
            {
                "dropped": k,
                "n": len(keep),
                "mean diff": keep["QUANT"].mean() - keep[baseline].mean(),
                "W/D/L": wdl(keep["QUANT"], keep[baseline]),
                "p": wilcoxon(keep["QUANT"], keep[baseline], zero_method="wilcox").pvalue,
                "last removed": "-" if k == 0 else gap.index[k - 1],
            }
        )
    return pd.DataFrame(rows).set_index("dropped")


print("QUANT vs DrCIF  (already p > 0.05 with all 112)")
display(leave_out("DrCIF").round(4))

print("\nQUANT vs rSTSF  (the one comparison significant before correction)")
display(leave_out("RSTSF").round(4))

QUANT vs DrCIF  (already p > 0.05 with all 112)


,n,mean diff,W/D/L,p,last removed
dropped,,,,,
0,112,0.0035,65/4/43,0.1595,-
1,111,0.0017,64/4/43,0.2081,PigAirwayPressure
2,110,0.0012,63/4/43,0.2679,UMD
3,109,0.0007,62/4/43,0.3400,EthanolLevel
4,108,0.0012,62/4/42,0.2550,ToeSegmentation1
5,107,0.0007,61/4/42,0.3253,PigArtPressure
6,106,0.0012,61/4/41,0.2420,Phoneme
7,105,0.0016,61/4/40,0.1738,Rock
8,104,0.0020,61/4/39,0.1202,FaceFour



QUANT vs rSTSF  (the one comparison significant before correction)


,n,mean diff,W/D/L,p,last removed
dropped,,,,,
0,112,0.0031,65/5/42,0.0294,-
1,111,0.0021,64/5/42,0.0413,PigAirwayPressure
2,110,0.0031,64/5/41,0.0251,InlineSkate
3,109,0.0024,63/5/41,0.0356,EOGHorizontalSignal
4,108,0.0032,63/5/40,0.0213,Rock
5,107,0.0025,62/5/40,0.0304,EthanolLevel
6,106,0.0031,62/5/39,0.0178,PigCVP
7,105,0.0027,61/5/39,0.0256,Car
8,104,0.0024,60/5/39,0.0366,ScreenType


**Against DrCIF**, removing `PigAirwayPressure` alone halves the mean difference, from +0.0035 to +0.0017. A single dataset out of 112 accounts for about half the reported gap.

**Against rSTSF**, the mean difference is more stable, but the significance is not: p rises from 0.0294 past the 0.05 threshold once nine datasets are removed (0.0515 at nine, 0.0717 at ten). QUANT's only win over a strong interval baseline that is significant before correction therefore depends on roughly 8% of the archive.

This is a quantified version of the caveat the authors themselves raise.

## 7. Robustness of our own choices

Two decisions in this notebook could have shaped the result: the `AUDIO` label we assigned to `InsectWingbeatSound`, and the inclusion of very small categories. Repeat section 4 without them.

In [9]:
MIN_N = 5

strict = df.drop(index=[d for d in JUDGEMENT_CALL if d in df.index])
counts = strict["Type"].value_counts()
strict = strict[strict["Type"].isin(counts[counts >= MIN_N].index)]

print(f"{len(strict)} datasets in categories with n >= {MIN_N}, "
      f"judgement-call labels excluded\n")

rows = []
for type_, g in strict.groupby("Type"):
    rows.append(
        {
            "type": type_,
            "n": len(g),
            "QUANT": g["QUANT"].mean(),
            "vs strong": np.mean([g["QUANT"].mean() - g[m].mean() for m in STRONG]),
            "wdl DrCIF": wdl(g["QUANT"], g["DrCIF"]),
            "p vs DrCIF": wilcoxon(g["QUANT"], g["DrCIF"], zero_method="wilcox").pvalue,
        }
    )

pd.DataFrame(rows).set_index("type").sort_values(
    "vs strong", ascending=False
).round(4)

102 datasets in categories with n >= 5, judgement-call labels excluded



,n,QUANT,vs strong,wdl DrCIF,p vs DrCIF
type,,,,,
DEVICE,9,0.8476,0.0119,7/0/2,0.0547
SIMULATED,9,0.9592,0.0042,7/0/2,0.3008
SENSOR,16,0.9208,0.0029,8/1/7,0.6496
HAR,11,0.8824,0.0024,5/1/5,0.4922
IMAGE,32,0.8453,0.0021,18/0/14,0.5608
ECG,7,0.9500,0.0006,4/0/3,1.0000
SPECTRO,12,0.8937,-0.0040,7/0/5,0.6772
MOTION,6,0.7318,-0.0159,3/0/3,0.5625


Across the eight categories large enough to analyse, no domain shows a significant difference from DrCIF. The closest is DEVICE at p = 0.0547 with 7/0/2 — the one place QUANT looks genuinely ahead, though nine datasets is too thin to settle it, and it would be the natural candidate for follow-up. The conclusion is unchanged: there is no domain in which QUANT demonstrably beats the strongest interval baselines.

## 8. Summary

1. **The assembled table reproduces Fig. 1 exactly** — identical means and win/draw/loss, and identical p-values under `zero_method="pratt"` — so the re-analysis is faithful to the published results.
2. **QUANT does not significantly outperform DrCIF.** The paper's prose describes QUANT as "more accurate" than DrCIF; its mean is higher, but the paper's own test (p = 0.1442) does not find the difference significant. The abstract's weaker claim of *the same accuracy* is what the evidence supports.
3. **The advantage is concentrated, not general — and nowhere significant.** Across IMAGE, SENSOR and SPECTRO (60 of 112 datasets) QUANT sits at parity with DrCIF, and it is behind on MOTION. Restricting to the eight categories with at least five datasets, no domain shows a significant difference from DrCIF; the largest apparent effect, DEVICE at +0.0119, is borderline (p = 0.0547, n = 9).
4. **One dataset carries half the gap.** Removing `PigAirwayPressure`, a 52-class problem on which both methods score poorly, halves the mean difference against DrCIF.
5. **The one win significant before correction is fragile.** Against rSTSF, p crosses 0.05 after dropping nine of 112 datasets.
6. **Our own grouping choice manufactured a spurious finding.** Merging four small physiological categories produced an apparent domain strength that disappears when the raw labels are used.

None of this undermines QUANT's actual contribution, which is accuracy *per unit compute* — it is three orders of magnitude faster than HC2. It does suggest the accuracy comparison against the strongest interval methods should be read as parity rather than superiority.

### Caveats

- p-values here use scipy's default `zero_method="wilcox"`, which drops zero differences. The paper's values correspond to `"pratt"`; §2 shows both.
- **STSF**, the fifth method in Fig. 1, is absent from aeon's results and is omitted throughout.
- Several raw categories have fewer than five datasets; section 7 repeats the analysis without them.
- Per-category tests below about ten datasets have very little power: a
  two-sided Wilcoxon at n = 6 cannot return a p-value below 0.031 even if
  every dataset favours one method, so "no significant difference" in the
  smallest categories is partly a statement about the test.
- `InsectWingbeatSound` was assigned to AUDIO by judgement, not from a source; section 7 excludes it.
- Domain labels are coarse and some assignments are debatable (`GunPoint` is labelled HAR rather than MOTION).
- The leave-out analysis is **descriptive, not a hypothesis test**. Dropping the most extreme points will always shrink a difference. It shows how many datasets a result rests on, not that the result is wrong.